# ⚡ 2x-5x Faster Multilingual RAG Fine-Tuning with Unsloth (Free Google Colab)

This notebook uses **Unsloth** to fine-tune **Qwen2.5-0.5B-Instruct** on our complete Hindi, Tamil, and English RAG dataset (~126k triplets).

### 🚀 Key Advantages of Unsloth:
- **2x–5x faster training**
- **80% less memory usage**
- **100% Free** on Google Colab T4 GPU (~10 to 15 minutes total)
- **1-Click Export** to merged weights and GGUF/Safetensors

In [ ]:
# Step 1: Install Unsloth and Dependencies (Optimized for Free Colab T4)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets triton

In [ ]:
# Step 2: Check Uploaded Dataset
import os
if not os.path.exists('rag_sft_dataset.jsonl'):
    print('⚠️ Please drag and drop rag_sft_dataset.jsonl into the Files panel on the left.')
else:
    size_mb = os.path.getsize('rag_sft_dataset.jsonl') / (1024 * 1024)
    print(f'✅ Dataset found! Size: {size_mb:.2f} MB')

In [ ]:
# Step 3: Load Qwen2.5-0.5B with Unsloth Fast Model Loading
from unsloth import FastLanguageModel
import torch

max_seq_length = 384
dtype = None  # None for auto detection (Float16 on T4)
load_in_4bit = True  # Uses 4bit quantization to drastically reduce VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Apply Unsloth Optimized LoRA (q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less memory
    random_state=3407,
)
print('✅ Model loaded with Unsloth LoRA optimizations!')

In [ ]:
# Step 4: Load & Format Dataset with Qwen Chat Template
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = load_dataset("json", data_files="rag_sft_dataset.jsonl", split="train")
dataset = dataset.shuffle(seed=42).map(formatting_prompts_func, batched=True)
print(f'✅ Formatted {len(dataset)} examples for training!')

In [ ]:
# Step 5: Fast Training with SFTTrainer & Unsloth Kernels
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=2,
        warmup_steps=100,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print('🚀 Starting Unsloth training on free GPU...')
trainer_stats = trainer.train()
print('✅ Training finished successfully!')

In [ ]:
# Step 6: Save & Export Merged 16-bit Model
model_save_dir = "qwen2.5_0.5b_indic_rag_merged"
model.save_pretrained_merged(model_save_dir, tokenizer, save_method="merged_16bit")

# Zip for 1-click direct download
!zip -r qwen2.5_0.5b_indic_rag_merged.zip qwen2.5_0.5b_indic_rag_merged
print('🎉 Finished! Download qwen2.5_0.5b_indic_rag_merged.zip from the Files panel on the left.')

In [ ]:
# Step 7 (Optional): Push directly to Hugging Face Hub
# from huggingface_hub import login
# login() # Paste your HF write token
# model.push_to_hub_merged("ansh123456789/qwen2.5-0.5b-indic-rag", tokenizer, save_method="merged_16bit")